# Preprocessing, seen

From a raw DICOM slice to the array the encoder receives, one transformation at a time.

**Nothing is implemented here.** Every figure comes from `rsna.viz`, and
`verify_against_read_slot` asserts the illustrated chain ends exactly where the real
pipeline ends — so these pictures cannot drift into showing a preprocessing nobody
runs.

Outputs are stripped on commit (`scripts/nbstrip.py`): rendered slices are competition
data, and rule 2.4.b forbids redistributing it. Run the cells to see the images.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks"
                else str(Path.cwd() / "src"))

import pandas as pd
%matplotlib inline

from rsna.config import Config
from rsna.dicom import annotate, laterality_of, pick_slots, walk
from rsna import viz

ROOT = Path("data/raw") if Path("data/raw").is_dir() else Path("../data/raw")
SPLIT = "test_series"

config = Config()
headers = annotate(walk(ROOT, SPLIT))
series_csv = pd.read_csv(ROOT / f"{SPLIT.replace('_series', '')}_series.csv")
plane_map = dict(zip(series_csv.SeriesInstanceUID, series_csv.Anatomical_Plane))
headers["plane"] = headers["SeriesInstanceUID"].map(plane_map)

slots = pick_slots(headers, plane_map, config)
sides, _ = laterality_of(headers, config)

print(f"{len(slots)} studies, {sum(len(s) for s in slots.values())} filled slots")
for study, filled in slots.items():
    print(f"  ...{study[-12:]}  side {sides.get(study) or '?'}  {sorted(filled)}")

## Pick one (study, slot)

Change these two lines and re-run everything below.

In [ ]:
STUDY = [s for s in slots if slots[s]][0]      # or paste a full StudyInstanceUID
SLOT = sorted(slots[STUDY])[0]                # e.g. "SAG_FLUID_FS"

record = slots[STUDY][SLOT]
record["plane"] = plane_map.get(record["SeriesInstanceUID"], "")
side = sides.get(STUDY)

viz.verify_against_read_slot(record, config)
print(f"{SLOT}  {record['SeriesDescription']!r}  {record['n_slices']} slices  "
      f"{record['px']:.3f} mm/px  side {side or 'unresolved'}")
print("illustrated chain matches read_slot exactly")

## 1. The stack

Every slice in **physical** order — not file order. Red = the ones sampled, across the
central 20–80% band. Note how much is discarded.

In [ ]:
fig = viz.figure_stack(record, config)

## 2. One slice, step by step

Raw → cropped to a constant 130 mm of anatomy → normalised by the stack's 1st–99th
percentile → resized → quantised to bytes → mirrored if this is a right knee.

The crop is in **millimetres**, so its size in pixels differs per series while the
physical extent does not. That is what makes two series comparable.

In [ ]:
fig = viz.figure_steps(record, config, side)

## 3. What the encoder actually receives

Three slices become the three channels of one RGB image — the "2.5D" trick.

The bottom row repeats it with the slices in **file-name order**, which is what a
pipeline that sorts by filename would feed the model. A SOP Instance UID is unique by
construction and ordered by nothing.

Watch the composites: neighbouring slices agree, so the RGB is nearly grey. Arbitrary
slices disagree, and the colour fringing *is* the noise the model would be trained
on — with no error raised anywhere.

In [ ]:
fig = viz.figure_channels(record, config, side)